# Различные типы А/Б тестов и их применение

## 1. Критерий хи-квадрат
### Когда использовать: 
Критерий хи-квадрат используется для категориальных данных с двумя или более категориями, например, для данных, которые выражают частоту или долю (успех/неуспех).
### Применение: 
Критерий хи-квадрат проверяет, отличается ли распределение категориальных данных в двух или более группах. Он особенно полезен для анализа таблиц сопряжённости (контингентных таблиц), в которых мы наблюдаем частоту случаев для каждой категории.
### Пример: 
Вы хотите проверить, различаются ли доли пользователей, которые совершили покупку (успех/неуспех), между двумя группами (например, между пользователями, которые видели A или B версию страницы).
### Предположения:
Выборка достаточно велика.
Частота в каждой категории должна быть не слишком малой (обычно ожидаемое значение в каждой ячейке должно быть не меньше 5).

In [2]:
from scipy.stats import chi2_contingency

# Таблица сопряженности для двух групп
# Количество успехов и неудач в двух группах
data = [[50, 150],  # Группа A (успехи, неудачи)
        [30, 170]]  # Группа B (успехи, неудачи)

# Применение критерия хи-квадрат
chi2, p_value, dof, expected = chi2_contingency(data)

print(f"Хи-квадрат статистика: {chi2}")
print(f"P-значение: {p_value}")


Хи-квадрат статистика: 5.640625
P-значение: 0.017548950191476703


### Расчет размера выборки

Для грубого расчета размера выборки можно использовать следующую формулу: 
$$
n=\frac{(Z_{\alpha/2}+Z_\beta)^2\cdot(p_1+p_2-\frac{1}{2}(p_1+p_2)^2)}{\Delta^2}
$$
$p_1$, $p_2$ - ожидаемые пропорции успеха в группах А и Б,

$\Delta$ - размер эффекта,

$Z_{\alpha / 2}$ - критическое значение, соответствующее искомому уровню значимости $\alpha / 2$. Берется $\alpha / 2$ потому, что мы рассматриваем двусторонний тест. Значение $\alpha$ определяет уровень допустимости ошибки первого рода. Как правило $\alpha$ берется 0.05 для обычных исследовний и 0.01 для случая медицинских исследований или иных, где ошибка первого рода (отвергнута верная нулевая гипотеза) является критически важной.

$Z_\beta$ - критическое значение, соответствующее заданному уровню статистической мощности. Значение $\beta$ определяет вероятность ошибки второго рода (принята неверная нулевая гипотеза). Как правило $\beta$ устанавливается на уровне 0.8 или 0.9.

#### Вычисляем вручную

In [4]:
from scipy import stats
import numpy as np

In [6]:
p_experiment = 0.02
p_control = 0.01
effect_size = p_experiment - p_control
confidence_level = 0.95
power = 0.8

z_alpha = stats.norm.ppf(1 - (1 - confidence_level) / 2)
print(f'Z_alpha / 2: {np.round(z_alpha, 2)}')
z_beta = abs(stats.norm.ppf(1 - power))
print(f'Z_beta: {np.round(z_beta, 2)}')

sample_size = (((z_alpha + z_beta)**2 * (p_control + p_experiment - 0.5*(p_experiment+p_control)**2))/effect_size)**2

# Округляем до целого значения
sample_size = int(sample_size) + 1

print(f"Минимальный размер выборки для каждой группы: {sample_size}")

Z_alpha / 2: 1.96
Z_beta: 0.84
Минимальный размер выборки для каждой группы: 538


#### Вычисляем с помощью пакета statsmodels

In [29]:
import statsmodels.api as sm

In [32]:
# Конверсии в процентах для двух групп
conversion_rate_experiment = 0.02  # Группа эксперимента
conversion_rate_control = 0.01  # Группа контроля

effect_size = sm.stats.proportion_effectsize(conversion_rate_control, conversion_rate_experiment)
stat_size = sm.stats.GofChisquarePower().solve_power(effect_size=effect_size, alpha=0.05, power=0.8)

print(f"Необходимое число наблюдений для каждой группы: {stat_size/2:.0f}")

Необходимое число наблюдений для каждой группы: 563


In [31]:
sm.stats.chisquare_effectsize(conversion_rate_control, conversion_rate_experiment)

0.0

## 2. Z-критерий
### Когда использовать:
Z-критерий используется для дискретных данных (доля или пропорция) в случае больших выборок. Он применяется для сравнения долей успехов между двумя группами, когда выборка достаточно велика, чтобы можно было аппроксимировать распределение нормальным.
### Применение: 
Z-критерий для пропорций оценивает, различаются ли доли успехов (например, покупок) между двумя группами.
### Пример: 
Вы хотите проверить, различаются ли доли пользователей, совершивших покупку в двух группах, при условии, что в каждой группе достаточно наблюдений.
### Предположения:
- Выборки достаточно велики.
- Данные представляют собой доли или пропорции.

In [3]:
from statsmodels.stats.proportion import proportions_ztest

# Данные для двух групп
successes = [50, 30]  # Количество покупок (успехов) в группе A и B
trials = [200, 180]   # Общее количество пользователей в группе A и B

# Применение Z-критерия для сравнения долей
z_stat, p_value = proportions_ztest(successes, trials)

print(f"Z-статистика: {z_stat}")
print(f"P-значение: {p_value}")


Z-статистика: 1.9895560643855537
P-значение: 0.046639860079136314


### Размер выборки

$$n = \Bigg( \frac{Z_{\alpha / 2} \sqrt{2\cdot p(1-p)} + Z_{\beta} \sqrt{p_1 \cdot (1-p_1)+ p_2(1-p_2)}}{\delta} \Bigg)^2$$

$Z_{\alpha / 2}$ - критическое значение, соответствующее искомому уровню значимости $\alpha / 2$. Берется $\alpha / 2$ потому, что мы рассматриваем двусторонний тест. Значение $\alpha$ определяет уровень допустимости ошибки первого рода. Как правило $\alpha$ берется 0.05 для обычных исследовний и 0.01 для случая медицинских исследований или иных, где ошибка первого рода (отвергнута верная нулевая гипотеза) является критически важной.

$Z_\beta$ - критическое значение, соответствующее заданному уровню статистической мощности. Значение $\beta$ определяет вероятность ошибки второго рода (принята неверная нулевая гипотеза). Как правило $\beta$ устанавливается на уровне 0.8 или 0.9.

$\delta = |p_1-p_2|$ - размер эффекта, который необходимо выловить

$p_1$ - базовый показатель конверсии

$p_2$ - показатель конверсии после изменений $= p_1 + \delta$

$p = \frac{p_1+p_2}{2}$ - средняя конверсия

#### Вычислим вручную

In [20]:
p_experiment = 0.02
p_control = 0.01
p_avg = (p_control + p_experiment)/2
effect_size = p_experiment - p_control
confidence_level = 0.95
power = 0.8

z_alpha = stats.norm.ppf(1 - (1 - confidence_level) / 2)
z_beta = abs(stats.norm.ppf(1 - power))

sample_size = ((z_alpha*np.sqrt(2*p_avg*(1-p_avg)) + z_beta*np.sqrt(p_control*(1-p_control)+p_experiment*(1-p_experiment)))/effect_size)**2

# Округляем до целого значения
sample_size = int(sample_size) + 1

print(f"Минимальный размер выборки для каждой группы: {sample_size}")

Минимальный размер выборки для каждой группы: 2319


#### Вычислим с помощью пакета statsmodels

In [23]:
# Конверсии в процентах для двух групп
conversion_rate_experiment = 0.02  # Группа эксперимента
conversion_rate_control = 0.01  # Группа контроля

effect_size = sm.stats.proportion_effectsize(conversion_rate_control, conversion_rate_experiment)
analysis = sm.stats.NormalIndPower()
stat_size = analysis.solve_power(effect_size=effect_size, alpha=0.05, power=0.8)

print(f"Необходимое число наблюдений: {stat_size:.0f}")

Необходимое число наблюдений: 2254


## 3. t-критерий
### Когда использовать: 
t-критерий используется для непрерывных количественных данных (например, доход, сумма покупки) и применяется для сравнения средних значений двух групп. t-тест наиболее полезен, когда выборки небольшие или когда мы не знаем дисперсии генеральной совокупности.
### Применение: 
t-критерий проверяет, различаются ли средние значения в двух выборках.
### Пример: 
Вы хотите проверить, различаются ли средние суммы покупок пользователей, которые видели A и B версии страницы.
### Предположения:
- Данные в каждой выборке распределены нормально (особенно важно для малых выборок).
- Группы независимы друг от друга.
- Дисперсии выборок равны (если они сильно различаются, рекомендуется использовать модифицированный тест Уэлча).

In [4]:
from scipy.stats import ttest_ind

# Пример данных о среднем чеке пользователей в двух группах
group_A = [23.4, 26.5, 21.7, 27.8, 24.2]  # Чеки пользователей, видевших версию A
group_B = [22.1, 23.4, 19.6, 20.8, 21.9]  # Чеки пользователей, видевших версию B

# Применение t-критерия для независимых выборок
t_stat, p_value = ttest_ind(group_A, group_B)

print(f"t-статистика: {t_stat}")
print(f"P-значение: {p_value}")


t-статистика: 2.4981993515330227
P-значение: 0.03704583537325475


### Размер выборки

Размер выборки для t-критерия рассчитывается следующим образом:
$$
n=\Bigg(\frac{Z_{\alpha}/2 + Z_\beta}{\delta/\sigma} \Bigg)^2
$$

$Z_{\alpha / 2}$ - критическое значение, соответствующее искомому уровню значимости $\alpha / 2$. Берется $\alpha / 2$ потому, что мы рассматриваем двусторонний тест. Значение $\alpha$ определяет уровень допустимости ошибки первого рода. Как правило $\alpha$ берется 0.05 для обычных исследовний и 0.01 для случая медицинских исследований или иных, где ошибка первого рода (отвергнута верная нулевая гипотеза) является критически важной.

$Z_\beta$ - критическое значение, соответствующее заданному уровню статистической мощности. Значение $\beta$ определяет вероятность ошибки второго рода (принята неверная нулевая гипотеза). Как правило $\beta$ устанавливается на уровне 0.8 или 0.9.

$\delta$ - размер эффекта, который необходимо выловить

#### Вычислим вручную

In [37]:
p_experiment = 0.02
p_control = 0.01
p_avg = (p_control + p_experiment)/2
effect_size = p_experiment - p_control
confidence_level = 0.95
power = 0.8
sigma = 1

z_alpha = stats.norm.ppf(1 - (1 - confidence_level) / 2)
z_beta = abs(stats.norm.ppf(1 - power))

sample_size = ((z_alpha + z_beta)/effect_size/sigma)**2

# Округляем до целого значения
sample_size = int(sample_size) + 1

print(f"Минимальный размер выборки для каждой группы: {sample_size}")

Минимальный размер выборки для каждой группы: 78489


#### Вычислим с помощью пакета statsmodels

In [36]:
from statsmodels.stats.power import TTestIndPower

# Параметры
effect_size = 0.01  # Величина эффекта (например, разница в средних в единицах стандартного отклонения)
alpha = 0.05       # Уровень значимости
power = 0.8        # Мощность теста

# Расчёт размера выборки для t-теста для независимых выборок
analysis = TTestIndPower()
sample_size = analysis.solve_power(effect_size=effect_size, alpha=alpha, power=power)
print(f"Необходимый размер выборки для каждой группы: {sample_size/2:.0f}")

Необходимый размер выборки для каждой группы: 78489
